In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!rm -rf Cap3_MedicionPobreza
!git clone https://github.com/LizamaD/Cap3_MedicionPobreza.git

Cloning into 'Cap3_MedicionPobreza'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 187 (delta 106), reused 146 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 4.40 MiB | 60.07 MiB/s, done.
Resolving deltas: 100% (106/106), done.


In [3]:
import pandas as pd

path_bases = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"
path_pobreza = '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Base final/pobreza_24.csv'


# --- Carga tus datos crudos ---
poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")
trabajos_raw = pd.read_csv(f"{path_bases}trabajos.csv")
gastospersona_raw = pd.read_csv(f"{path_bases}gastospersona.csv")
gastoshogar_raw = pd.read_csv(f"{path_bases}gastoshogar.csv")
ingresos_raw = pd.read_csv(f"{path_bases}ingresos.csv")
pobreza_raw = pd.read_csv(path_pobreza)

/tmp/ipykernel_10838/1323284465.py:8: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
/tmp/ipykernel_10838/1323284465.py:9: DtypeWarning: Columns (1,4,27,44) have mixed types. Specify dtype option on import or set low_memory=False.
  viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
/tmp/ipykernel_10838/1323284465.py:10: DtypeWarning: Columns (45,49,53,61,63,69,75,77,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")


In [4]:
%cd /content/Cap3_MedicionPobreza

/content/Cap3_MedicionPobreza


In [5]:
from src.trabajos import process_trabajos
from src.ingresos import generar_ingreso_deflactado_ago2024
from src.gastoshogar import procesar_gastos_enigh
from src.gastospersona import procesar_gastos_persona_enigh
from src.hogares import process_hogares
from src.poblacion import process_poblacion
from src.viviendas import process_viviendas
from src.pipeline import create_master_table, impute_data, prepare_and_split_for_autoencoder

In [ ]:
# --- Crea la tabla maestra ---
# Esta función procesa, une todo y guarda el resultado en "master_table.csv"
master_df = create_master_table(
    pob_keys=pobreza_raw,
    pob_df=poblacion_raw,
    viv_df=viviendas_raw,
    hog_df=hogares_raw,
    trab_df=trabajos_raw,
    gasper_df=gastospersona_raw,
    gashog_df=gastoshogar_raw,
    ing_df=ingresos_raw,
    output_path=f"{path_bases}enigh_master_table.csv" # Ruta sugerida
)

# --- Imputa los datos faltantes ---
# Esta función toma la tabla maestra y la deja lista para el modelado
df_final_limpio = impute_data(master_df)

# --- Verifica el resultado ---
print("\nDataFrame final después de la imputación:")
print(df_final_limpio.head())

# Confirma que no hay valores nulos
print("\nSuma de nulos por columna en el DF final:")
print(df_final_limpio.isnull().sum().sum()) # Debería ser 0


Procesando tablas individuales...


In [ ]:
# Define tu ruta de salida en Google Drive
output_directory = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"

# Llama a la nueva función para procesar y guardar todo
df_processed, df_pobres, df_no_pobres = prepare_and_split_for_autoencoder(
    df_final_limpio, 
    output_directory
)

# Ahora ya tienes tus DataFrames listos para la estrategia de descubrimiento:
# - df_no_pobres: Para entrenar y validar tu autoencoder.
# - df_pobres: Para pasarlo por el modelo ya entrenado y analizar el error de reconstrucción.



Iniciando preparación y split para el autoencoder...
Convirtiendo 17 columnas categóricas a formato dummy...
El DataFrame ahora tiene 328 columnas después del one-hot encoding.
Guardando el DataFrame procesado completo en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos//full_processed.csv'...
Realizando el split entre hogares pobres y no pobres...
  - Encontrados 90017 registros de personas en pobreza.
  - Encontrados 218427 registros de personas no pobres.
Guardando el set de pobres en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos//pobres.csv'...
Guardando el set de no pobres en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos//no_pobres.csv'...
Proceso completado. ¡Archivos listos para el modelado!
